# Landslide source/runout step 10: protected status of existing forest associated with protection benefits

This notebook estimates how much **existing forest associated with landslide protection benefits** falls inside Jamaica forest reserves/protected areas versus outside them.

The method is deliberately area-based rather than a dollar attribution:

1. Load the projected source/runout rasters used in the source-and-runout landslide workflow.
2. For each return period, identify cells where `deforestation class > baseline class`. These are cells where existing forest reduces the source/runout hazard class relative to the deforestation counterfactual.
3. Intersect those class-reduction cells with existing forest from the forest-equivalent land-cover layer.
4. Split the resulting forest area by protected status using forest reserves and protected areas.

Important interpretation:
- This identifies forest areas spatially associated with hazard-class reductions that drive the protection scenario.
- It does **not** causally allocate individual avoided-EAD dollars to forest polygons. That would require an additional asset-to-forest attribution model.
- The main quantities are forest footprint area, forest-equivalent area, and class-reduction-weighted forest-equivalent area.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize

pd.options.display.max_columns = 120
pd.options.display.float_format = '{:,.3f}'.format

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
landslide_root = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages'
output_dir = landslide_root / 'protected_forest_attribution_source_and_runout_zones'
output_dir.mkdir(parents=True, exist_ok=True)

raster_dir = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/landslide_source_and_runout_zones_network_intersections/projected_source_and_runout_rasters_epsg3448'
landuse_path = base_path / 'dphil_paper_2/processed_data/land_use_forest_and_afforestable.gpkg'
forest_reserves_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/protected_landcover/forest_reserves.shp'
protected_areas_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/protected_landcover/protected_areas.shp'

return_periods = [5, 10, 25, 50, 100]
reference_raster = raster_dir / 'landslide_source_and_runout_baseline_rp_5_epsg3448.tif'

for required_path in [reference_raster, landuse_path, forest_reserves_path, protected_areas_path]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print('Output directory:', output_dir)
print('Raster directory:', raster_dir)
print('Land-use forest layer:', landuse_path)
print('Forest reserves:', forest_reserves_path)
print('Protected areas:', protected_areas_path)

## Load reference raster metadata

In [ ]:
with rasterio.open(reference_raster) as src:
    reference_crs = src.crs
    reference_transform = src.transform
    reference_shape = (src.height, src.width)
    reference_bounds = src.bounds
    pixel_area_m2 = abs(src.transform.a * src.transform.e)

if reference_crs is None:
    raise ValueError('Reference raster has no CRS.')

print('Reference CRS:', reference_crs)
print('Reference shape:', reference_shape)
print('Reference bounds:', reference_bounds)
print(f'Pixel area: {pixel_area_m2:,.3f} m² ({pixel_area_m2 / 10_000:,.5f} ha)')

## Load existing forest and protected-area geometry

`forest_flood_equivalent_values` is used because this is the existing forest-equivalent layer already used elsewhere in the DPhil workflow:

- `1.0` = existing forest / tree-cover equivalent;
- `0.5` = mixed field/forest or field/bamboo classes;
- `0.0` = not existing forest for this analysis.

The notebook reports both forest footprint area and forest-equivalent area.

In [ ]:
target_crs = reference_crs

landuse = gpd.read_file(landuse_path, layer='land_use')
if landuse.crs is None:
    raise ValueError('Land-use layer has no CRS.')
if landuse.crs != target_crs:
    landuse = landuse.to_crs(target_crs)

if 'forest_flood_equivalent_values' not in landuse.columns:
    raise KeyError("Land-use layer missing 'forest_flood_equivalent_values'.")

forest = landuse.loc[
    pd.to_numeric(landuse['forest_flood_equivalent_values'], errors='coerce').fillna(0.0) > 0,
    ['Classify', 'LU_CODE', 'forest_flood_equivalent_values', 'geometry']
].copy()
forest['Forest_Equivalent_Value'] = pd.to_numeric(forest['forest_flood_equivalent_values'], errors='coerce').fillna(0.0)
forest = forest[forest.geometry.notna()].copy()
forest = forest[~forest.geometry.is_empty].copy()
forest['geometry'] = forest.geometry.make_valid()

forest_reserves = gpd.read_file(forest_reserves_path)
protected_areas = gpd.read_file(protected_areas_path)

for name, gdf in [('forest_reserves', forest_reserves), ('protected_areas', protected_areas)]:
    if gdf.crs is None:
        raise ValueError(f'{name} has no CRS.')

if forest_reserves.crs != target_crs:
    forest_reserves = forest_reserves.to_crs(target_crs)
if protected_areas.crs != target_crs:
    protected_areas = protected_areas.to_crs(target_crs)

forest_reserves = forest_reserves[forest_reserves.geometry.notna()].copy()
forest_reserves = forest_reserves[~forest_reserves.geometry.is_empty].copy()
forest_reserves['geometry'] = forest_reserves.geometry.make_valid()

protected_areas = protected_areas[protected_areas.geometry.notna()].copy()
protected_areas = protected_areas[~protected_areas.geometry.is_empty].copy()
protected_areas['geometry'] = protected_areas.geometry.make_valid()

print(f'Existing forest polygons: {len(forest):,}')
print(f'Forest reserve polygons: {len(forest_reserves):,}')
print(f'Protected-area polygons: {len(protected_areas):,}')

forest_class_summary = (
    forest.assign(Area_ha=forest.geometry.area / 10_000.0)
    .groupby(['Classify', 'LU_CODE', 'Forest_Equivalent_Value'], dropna=False, as_index=False)
    .agg(Polygon_Count=('Classify', 'size'), Footprint_Area_ha=('Area_ha', 'sum'))
    .sort_values(['Forest_Equivalent_Value', 'Footprint_Area_ha'], ascending=[False, False])
)
forest_class_summary['Forest_Equivalent_Area_ha'] = forest_class_summary['Footprint_Area_ha'] * forest_class_summary['Forest_Equivalent_Value']
forest_class_summary_file = output_dir / 'existing_forest_classes_used_for_landslide_protection_attribution.csv'
forest_class_summary.to_csv(forest_class_summary_file, index=False)
print('Saved:', forest_class_summary_file)
display(forest_class_summary)

## Rasterize existing forest and protected status

Rasterization uses the source/runout raster grid. The default centre-of-cell rule is used (`all_touched=False`) to avoid overestimating narrow boundary overlaps.

In [ ]:
def rasterize_shapes(gdf, value_col=None, burn_value=1, dtype='uint8'):
    if gdf.empty:
        return np.zeros(reference_shape, dtype=dtype)
    if value_col is None:
        shapes = ((geom, burn_value) for geom in gdf.geometry if geom is not None and not geom.is_empty)
    else:
        shapes = (
            (geom, float(value))
            for geom, value in zip(gdf.geometry, gdf[value_col])
            if geom is not None and not geom.is_empty
        )
    return rasterize(
        shapes,
        out_shape=reference_shape,
        transform=reference_transform,
        fill=0,
        dtype=dtype,
        all_touched=False,
    )


forest_equivalent = rasterize_shapes(forest, value_col='Forest_Equivalent_Value', dtype='float32')
forest_footprint = forest_equivalent > 0

forest_reserve_mask = rasterize_shapes(forest_reserves, dtype='uint8') > 0
protected_area_mask = rasterize_shapes(protected_areas, dtype='uint8') > 0

# 0: outside both; 1: forest reserve only; 2: protected area only; 3: both.
protected_status_code = forest_reserve_mask.astype('uint8') + (2 * protected_area_mask.astype('uint8'))

status_lookup = {
    0: 'Outside forest reserves/protected areas',
    1: 'Forest reserve only',
    2: 'Protected area only',
    3: 'Forest reserve and protected area',
}

print(f'Forest footprint cells: {int(forest_footprint.sum()):,}')
print(f'Forest footprint area: {forest_footprint.sum() * pixel_area_m2 / 10_000:,.1f} ha')
print(f'Forest-equivalent area: {forest_equivalent.sum() * pixel_area_m2 / 10_000:,.1f} ha')
print('Protected status codes present:', sorted(np.unique(protected_status_code[forest_footprint]).tolist()))

## Summary helpers

In [ ]:
def format_area_ha(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.1f} ha'


def format_pct(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.1f}%'


def summarize_mask_by_status(mask, forest_weight=None, class_reduction=None, return_period=None, summary_type=''):
    if forest_weight is None:
        forest_weight = forest_equivalent
    rows = []
    for status_code, status_label in status_lookup.items():
        status_mask = mask & (protected_status_code == status_code)
        footprint_area_ha = float(status_mask.sum() * pixel_area_m2 / 10_000.0)
        forest_equiv_area_ha = float(forest_weight[status_mask].sum() * pixel_area_m2 / 10_000.0)
        if class_reduction is None:
            class_weighted_area_ha = np.nan
            mean_class_reduction = np.nan
        else:
            class_values = class_reduction[status_mask].astype('float32')
            forest_weights = forest_weight[status_mask].astype('float32')
            class_weighted_area_ha = float((class_values * forest_weights).sum() * pixel_area_m2 / 10_000.0)
            denom = float(forest_weights.sum())
            mean_class_reduction = float((class_values * forest_weights).sum() / denom) if denom > 0 else np.nan
        rows.append({
            'Summary_Type': summary_type,
            'ReturnPeriod': return_period,
            'Protected_Status_Code': status_code,
            'Protected_Status': status_label,
            'Forest_Footprint_Area_ha': footprint_area_ha,
            'Forest_Equivalent_Area_ha': forest_equiv_area_ha,
            'Class_Reduction_Weighted_Forest_Equivalent_Area_ha': class_weighted_area_ha,
            'Mean_Class_Reduction_Forest_Equivalent_Weighted': mean_class_reduction,
            'Cell_Count': int(status_mask.sum()),
        })
    out = pd.DataFrame(rows)
    for area_col in [
        'Forest_Footprint_Area_ha',
        'Forest_Equivalent_Area_ha',
        'Class_Reduction_Weighted_Forest_Equivalent_Area_ha',
    ]:
        total = out[area_col].sum(skipna=True)
        out[f'Pct_of_Total_{area_col}'] = np.where(total > 0, 100.0 * out[area_col] / total, 0.0)
    return out


def summarize_union_status(status_summary):
    protected = status_summary.loc[status_summary['Protected_Status_Code'] > 0].copy()
    outside = status_summary.loc[status_summary['Protected_Status_Code'] == 0].copy()
    rows = []
    for label, subset in [('Inside forest reserves/protected areas', protected), ('Outside forest reserves/protected areas', outside)]:
        row = {
            'Summary_Type': subset['Summary_Type'].iloc[0] if not subset.empty else None,
            'ReturnPeriod': subset['ReturnPeriod'].iloc[0] if not subset.empty else None,
            'Protected_Union_Status': label,
        }
        for col in [
            'Forest_Footprint_Area_ha',
            'Forest_Equivalent_Area_ha',
            'Class_Reduction_Weighted_Forest_Equivalent_Area_ha',
            'Cell_Count',
        ]:
            row[col] = float(subset[col].sum(skipna=True)) if col != 'Cell_Count' else int(subset[col].sum())
        if row['Forest_Equivalent_Area_ha'] > 0 and 'Mean_Class_Reduction_Forest_Equivalent_Weighted' in subset:
            weighted_sum = subset['Class_Reduction_Weighted_Forest_Equivalent_Area_ha'].sum(skipna=True)
            row['Mean_Class_Reduction_Forest_Equivalent_Weighted'] = weighted_sum / row['Forest_Equivalent_Area_ha']
        else:
            row['Mean_Class_Reduction_Forest_Equivalent_Weighted'] = np.nan
        rows.append(row)
    out = pd.DataFrame(rows)
    for area_col in [
        'Forest_Footprint_Area_ha',
        'Forest_Equivalent_Area_ha',
        'Class_Reduction_Weighted_Forest_Equivalent_Area_ha',
    ]:
        total = out[area_col].sum(skipna=True)
        out[f'Pct_of_Total_{area_col}'] = np.where(total > 0, 100.0 * out[area_col] / total, 0.0)
    return out


def add_readable_columns(df):
    out = df.copy()
    area_cols = [
        c for c in out.columns
        if c.endswith('_Area_ha') and not c.startswith('Pct_')
    ]
    pct_cols = [
        c for c in out.columns
        if (c.startswith('Pct_of_Total_') or c.startswith('Pct_'))
        and not c.endswith('_Readable')
        and not c.endswith('_Label')
    ]
    for col in area_cols:
        out[f'{col}_Readable'] = out[col].apply(format_area_ha)
    for col in pct_cols:
        out[f'{col}_Label'] = out[col].apply(format_pct)
    return out

## Existing forest protected-status baseline

This is the denominator: all existing forest-equivalent land-cover area, before restricting to cells where forest changes the landslide source/runout class.

In [ ]:
all_existing_forest_by_status = summarize_mask_by_status(
    forest_footprint,
    forest_weight=forest_equivalent,
    class_reduction=None,
    return_period=None,
    summary_type='all_existing_forest',
)
all_existing_forest_union = summarize_union_status(all_existing_forest_by_status)

all_existing_forest_by_status = add_readable_columns(all_existing_forest_by_status)
all_existing_forest_union = add_readable_columns(all_existing_forest_union)

existing_status_file = output_dir / 'existing_forest_by_protected_status_source_and_runout_grid.csv'
existing_union_file = output_dir / 'existing_forest_by_protected_union_status_source_and_runout_grid.csv'
all_existing_forest_by_status.to_csv(existing_status_file, index=False)
all_existing_forest_union.to_csv(existing_union_file, index=False)

print('Saved:', existing_status_file)
print('Saved:', existing_union_file)
display(all_existing_forest_by_status)
display(all_existing_forest_union)

## Identify existing-forest cells where deforestation increases landslide source/runout class

In [ ]:
rp_status_summaries = []
rp_union_summaries = []
class_change_rows = []
any_class_reduction = np.zeros(reference_shape, dtype='bool')
max_class_reduction = np.zeros(reference_shape, dtype='uint8')

for return_period in return_periods:
    baseline_path = raster_dir / f'landslide_source_and_runout_baseline_rp_{return_period}_epsg3448.tif'
    deforestation_path = raster_dir / f'landslide_source_and_runout_deforestation_rp_{return_period}_epsg3448.tif'
    for path in [baseline_path, deforestation_path]:
        if not path.exists():
            raise FileNotFoundError(path)

    with rasterio.open(baseline_path) as baseline_src, rasterio.open(deforestation_path) as deforestation_src:
        if baseline_src.shape != reference_shape or deforestation_src.shape != reference_shape:
            raise ValueError(f'Unexpected raster shape for RP {return_period}')
        baseline = baseline_src.read(1).astype('int16')
        deforestation = deforestation_src.read(1).astype('int16')

    class_reduction = np.maximum(deforestation - baseline, 0).astype('uint8')
    contribution_mask = forest_footprint & (class_reduction > 0)

    any_class_reduction |= contribution_mask
    max_class_reduction = np.maximum(max_class_reduction, class_reduction)

    status_summary = summarize_mask_by_status(
        contribution_mask,
        forest_weight=forest_equivalent,
        class_reduction=class_reduction,
        return_period=return_period,
        summary_type='forest_with_deforestation_minus_baseline_class_reduction',
    )
    union_summary = summarize_union_status(status_summary)
    rp_status_summaries.append(status_summary)
    rp_union_summaries.append(union_summary)

    for status_code, status_label in status_lookup.items():
        for change_value in sorted(np.unique(class_reduction[contribution_mask]).tolist()):
            if change_value <= 0:
                continue
            mask = contribution_mask & (protected_status_code == status_code) & (class_reduction == change_value)
            class_change_rows.append({
                'ReturnPeriod': return_period,
                'Protected_Status_Code': status_code,
                'Protected_Status': status_label,
                'Class_Reduction': int(change_value),
                'Forest_Footprint_Area_ha': float(mask.sum() * pixel_area_m2 / 10_000.0),
                'Forest_Equivalent_Area_ha': float(forest_equivalent[mask].sum() * pixel_area_m2 / 10_000.0),
                'Class_Reduction_Weighted_Forest_Equivalent_Area_ha': float((forest_equivalent[mask] * change_value).sum() * pixel_area_m2 / 10_000.0),
                'Cell_Count': int(mask.sum()),
            })

    print(
        f'RP {return_period}: contributing forest-equivalent area = '
        f'{status_summary["Forest_Equivalent_Area_ha"].sum():,.1f} ha; '
        f'class-reduction-weighted = {status_summary["Class_Reduction_Weighted_Forest_Equivalent_Area_ha"].sum():,.1f} ha-class'
    )

rp_status_summary = pd.concat(rp_status_summaries, ignore_index=True)
rp_union_summary = pd.concat(rp_union_summaries, ignore_index=True)
class_change_summary = pd.DataFrame(class_change_rows)

rp_status_summary = add_readable_columns(rp_status_summary)
rp_union_summary = add_readable_columns(rp_union_summary)
class_change_summary = add_readable_columns(class_change_summary)

rp_status_file = output_dir / 'landslide_protection_contributing_existing_forest_by_rp_and_protected_status.csv'
rp_union_file = output_dir / 'landslide_protection_contributing_existing_forest_by_rp_and_protected_union_status.csv'
class_change_file = output_dir / 'landslide_protection_contributing_existing_forest_by_rp_status_and_class_reduction.csv'

rp_status_summary.to_csv(rp_status_file, index=False)
rp_union_summary.to_csv(rp_union_file, index=False)
class_change_summary.to_csv(class_change_file, index=False)

print('Saved:', rp_status_file)
print('Saved:', rp_union_file)
print('Saved:', class_change_file)
display(rp_union_summary)

## Any-return-period contributing forest area

This is the main table for writing: forest cells that show a deforestation-minus-baseline class reduction for **at least one** return period.

In [ ]:
any_status_summary = summarize_mask_by_status(
    any_class_reduction,
    forest_weight=forest_equivalent,
    class_reduction=max_class_reduction,
    return_period='any_5_10_25_50_100',
    summary_type='forest_with_any_return_period_class_reduction',
)
any_union_summary = summarize_union_status(any_status_summary)

# Add comparison to all existing forest in the same status.
existing_by_status_lookup = all_existing_forest_by_status.set_index('Protected_Status_Code')
for area_col in ['Forest_Footprint_Area_ha', 'Forest_Equivalent_Area_ha']:
    any_status_summary[f'Pct_of_Status_Existing_{area_col}_Contributing'] = [
        100.0 * row[area_col] / existing_by_status_lookup.loc[row['Protected_Status_Code'], area_col]
        if existing_by_status_lookup.loc[row['Protected_Status_Code'], area_col] > 0 else 0.0
        for _, row in any_status_summary.iterrows()
    ]

existing_union_lookup = all_existing_forest_union.set_index('Protected_Union_Status')
for area_col in ['Forest_Footprint_Area_ha', 'Forest_Equivalent_Area_ha']:
    any_union_summary[f'Pct_of_Status_Existing_{area_col}_Contributing'] = [
        100.0 * row[area_col] / existing_union_lookup.loc[row['Protected_Union_Status'], area_col]
        if existing_union_lookup.loc[row['Protected_Union_Status'], area_col] > 0 else 0.0
        for _, row in any_union_summary.iterrows()
    ]

any_status_summary = add_readable_columns(any_status_summary)
any_union_summary = add_readable_columns(any_union_summary)

any_status_file = output_dir / 'landslide_protection_contributing_existing_forest_any_rp_by_protected_status.csv'
any_union_file = output_dir / 'landslide_protection_contributing_existing_forest_any_rp_by_protected_union_status.csv'
any_status_summary.to_csv(any_status_file, index=False)
any_union_summary.to_csv(any_union_file, index=False)

print('Saved:', any_status_file)
print('Saved:', any_union_file)
display(any_status_summary)
display(any_union_summary)

## Concise drafting values

In [ ]:
protected_any = any_union_summary.loc[
    any_union_summary['Protected_Union_Status'] == 'Inside forest reserves/protected areas'
].iloc[0]
outside_any = any_union_summary.loc[
    any_union_summary['Protected_Union_Status'] == 'Outside forest reserves/protected areas'
].iloc[0]

protected_existing = all_existing_forest_union.loc[
    all_existing_forest_union['Protected_Union_Status'] == 'Inside forest reserves/protected areas'
].iloc[0]
outside_existing = all_existing_forest_union.loc[
    all_existing_forest_union['Protected_Union_Status'] == 'Outside forest reserves/protected areas'
].iloc[0]

rp_share_range = (
    rp_union_summary
    .loc[rp_union_summary['Protected_Union_Status'] == 'Inside forest reserves/protected areas']
    .agg({
        'Pct_of_Total_Forest_Equivalent_Area_ha': ['min', 'max'],
        'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha': ['min', 'max'],
    })
)

summary_rows = [
    {
        'Metric': 'Any RP contributing forest-equivalent area inside reserves/protected areas',
        'Value': protected_any['Forest_Equivalent_Area_ha'],
        'Readable': format_area_ha(protected_any['Forest_Equivalent_Area_ha']),
        'Percent': protected_any['Pct_of_Total_Forest_Equivalent_Area_ha'],
        'Percent_Label': format_pct(protected_any['Pct_of_Total_Forest_Equivalent_Area_ha']),
    },
    {
        'Metric': 'Any RP contributing forest-equivalent area outside reserves/protected areas',
        'Value': outside_any['Forest_Equivalent_Area_ha'],
        'Readable': format_area_ha(outside_any['Forest_Equivalent_Area_ha']),
        'Percent': outside_any['Pct_of_Total_Forest_Equivalent_Area_ha'],
        'Percent_Label': format_pct(outside_any['Pct_of_Total_Forest_Equivalent_Area_ha']),
    },
    {
        'Metric': 'Any RP class-reduction-weighted forest-equivalent area inside reserves/protected areas',
        'Value': protected_any['Class_Reduction_Weighted_Forest_Equivalent_Area_ha'],
        'Readable': format_area_ha(protected_any['Class_Reduction_Weighted_Forest_Equivalent_Area_ha']),
        'Percent': protected_any['Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha'],
        'Percent_Label': format_pct(protected_any['Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha']),
    },
    {
        'Metric': 'All existing forest-equivalent area inside reserves/protected areas',
        'Value': protected_existing['Forest_Equivalent_Area_ha'],
        'Readable': format_area_ha(protected_existing['Forest_Equivalent_Area_ha']),
        'Percent': protected_existing['Pct_of_Total_Forest_Equivalent_Area_ha'],
        'Percent_Label': format_pct(protected_existing['Pct_of_Total_Forest_Equivalent_Area_ha']),
    },
    {
        'Metric': 'All existing forest-equivalent area outside reserves/protected areas',
        'Value': outside_existing['Forest_Equivalent_Area_ha'],
        'Readable': format_area_ha(outside_existing['Forest_Equivalent_Area_ha']),
        'Percent': outside_existing['Pct_of_Total_Forest_Equivalent_Area_ha'],
        'Percent_Label': format_pct(outside_existing['Pct_of_Total_Forest_Equivalent_Area_ha']),
    },
    {
        'Metric': 'RP range: protected share of contributing forest-equivalent area',
        'Value': np.nan,
        'Readable': 'NA',
        'Percent': np.nan,
        'Percent_Label': f"{rp_share_range.loc['min', 'Pct_of_Total_Forest_Equivalent_Area_ha']:,.1f}–{rp_share_range.loc['max', 'Pct_of_Total_Forest_Equivalent_Area_ha']:,.1f}%",
    },
    {
        'Metric': 'RP range: protected share of class-reduction-weighted contributing forest-equivalent area',
        'Value': np.nan,
        'Readable': 'NA',
        'Percent': np.nan,
        'Percent_Label': f"{rp_share_range.loc['min', 'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha']:,.1f}–{rp_share_range.loc['max', 'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha']:,.1f}%",
    },
]

drafting_values = pd.DataFrame(summary_rows)
drafting_values_file = output_dir / 'landslide_protected_forest_attribution_drafting_values.csv'
drafting_values.to_csv(drafting_values_file, index=False)
print('Saved:', drafting_values_file)
display(drafting_values)

## Charts

In [ ]:
# Chart 1: any-RP contributing forest-equivalent area by detailed protected status.
plot_any = any_status_summary.copy().sort_values('Forest_Equivalent_Area_ha', ascending=True)
fig, ax = plt.subplots(figsize=(10.5, 6.2))
ax.barh(
    plot_any['Protected_Status'],
    plot_any['Forest_Equivalent_Area_ha'],
    color=['#bdbdbd' if code == 0 else '#31a354' for code in plot_any['Protected_Status_Code']],
    edgecolor='#3a3a3a',
    linewidth=0.7,
)
ax.set_xlabel('Contributing existing forest-equivalent area (ha)')
ax.set_title('Existing forest associated with landslide protection benefits by protected status')
ax.grid(axis='x', alpha=0.25)
for idx, row in enumerate(plot_any.itertuples(index=False)):
    ax.text(row.Forest_Equivalent_Area_ha, idx, f' {row.Pct_of_Total_Forest_Equivalent_Area_ha:.1f}%', va='center', fontsize=9)
plt.tight_layout()
any_status_chart = output_dir / 'landslide_contributing_existing_forest_any_rp_by_protected_status.png'
fig.savefig(any_status_chart, dpi=300, bbox_inches='tight')
print('Saved:', any_status_chart)
plt.show()

# Chart 2: return-period protected vs outside shares.
plot_rp = rp_union_summary.copy()
pivot = plot_rp.pivot(index='ReturnPeriod', columns='Protected_Union_Status', values='Forest_Equivalent_Area_ha').reindex(return_periods)
pivot = pivot.fillna(0.0)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
bottom = np.zeros(len(pivot))
colors = {
    'Inside forest reserves/protected areas': '#31a354',
    'Outside forest reserves/protected areas': '#bdbdbd',
}
for col in ['Inside forest reserves/protected areas', 'Outside forest reserves/protected areas']:
    values = pivot[col].to_numpy()
    ax.bar(pivot.index.astype(str), values, bottom=bottom, label=col, color=colors[col], edgecolor='#3a3a3a', linewidth=0.6)
    bottom += values
ax.set_xlabel('Return period')
ax.set_ylabel('Contributing forest-equivalent area (ha)')
ax.set_title('Return-period forest area associated with protection class reductions')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=True, loc='upper right')
plt.tight_layout()
rp_chart = output_dir / 'landslide_contributing_existing_forest_by_rp_protected_union_status.png'
fig.savefig(rp_chart, dpi=300, bbox_inches='tight')
print('Saved:', rp_chart)
plt.show()

## Reafforestation/afforestable area protected-status check

This section answers whether the reafforestation opportunity area intersects forest reserves and protected areas. It uses `afforestable_values > 0` from the same forest/afforestable land-use layer and the same source/runout raster grid.

In [ ]:
afforestable = landuse.loc[
    pd.to_numeric(landuse['afforestable_values'], errors='coerce').fillna(0.0) > 0,
    ['Classify', 'LU_CODE', 'afforestable_values', 'geometry']
].copy()
afforestable['Afforestable_Value'] = pd.to_numeric(afforestable['afforestable_values'], errors='coerce').fillna(0.0)
afforestable = afforestable[afforestable.geometry.notna()].copy()
afforestable = afforestable[~afforestable.geometry.is_empty].copy()
afforestable['geometry'] = afforestable.geometry.make_valid()

afforestable_equivalent = rasterize_shapes(afforestable, value_col='Afforestable_Value', dtype='float32')
afforestable_footprint = afforestable_equivalent > 0

afforestable_status_summary = summarize_mask_by_status(
    afforestable_footprint,
    forest_weight=afforestable_equivalent,
    class_reduction=None,
    return_period=None,
    summary_type='all_reafforestation_afforestable_area',
)
afforestable_union_summary = summarize_union_status(afforestable_status_summary)

afforestable_status_summary = afforestable_status_summary.rename(columns={
    'Forest_Footprint_Area_ha': 'Afforestable_Footprint_Area_ha',
    'Forest_Equivalent_Area_ha': 'Afforestable_Equivalent_Area_ha',
    'Pct_of_Total_Forest_Footprint_Area_ha': 'Pct_of_Total_Afforestable_Footprint_Area_ha',
    'Pct_of_Total_Forest_Equivalent_Area_ha': 'Pct_of_Total_Afforestable_Equivalent_Area_ha',
})
afforestable_union_summary = afforestable_union_summary.rename(columns={
    'Forest_Footprint_Area_ha': 'Afforestable_Footprint_Area_ha',
    'Forest_Equivalent_Area_ha': 'Afforestable_Equivalent_Area_ha',
    'Pct_of_Total_Forest_Footprint_Area_ha': 'Pct_of_Total_Afforestable_Footprint_Area_ha',
    'Pct_of_Total_Forest_Equivalent_Area_ha': 'Pct_of_Total_Afforestable_Equivalent_Area_ha',
})

afforestable_status_summary = add_readable_columns(afforestable_status_summary)
afforestable_union_summary = add_readable_columns(afforestable_union_summary)

afforestable_status_file = output_dir / 'reafforestation_afforestable_area_by_protected_status_source_and_runout_grid.csv'
afforestable_union_file = output_dir / 'reafforestation_afforestable_area_by_protected_union_status_source_and_runout_grid.csv'
afforestable_status_summary.to_csv(afforestable_status_file, index=False)
afforestable_union_summary.to_csv(afforestable_union_file, index=False)

print('Saved:', afforestable_status_file)
print('Saved:', afforestable_union_file)
display(afforestable_status_summary)
display(afforestable_union_summary)

## Reafforestation/afforestable cells with source/runout class reductions

This is the stricter reafforestation-side equivalent of the protection attribution above. It filters the afforestable layer to cells where `baseline class > reafforestation class` for at least one return period. This is a hazard-reduction proxy for reafforestation areas associated with landslide avoided EADs, but it is still not a direct dollar attribution of avoided EADs to afforestable polygons.

In [ ]:
reafforestation_rp_status_summaries = []
reafforestation_rp_union_summaries = []
reafforestation_any_class_reduction = np.zeros(reference_shape, dtype='bool')
reafforestation_max_class_reduction = np.zeros(reference_shape, dtype='uint8')

for return_period in return_periods:
    baseline_path = raster_dir / f'landslide_source_and_runout_baseline_rp_{return_period}_epsg3448.tif'
    reafforestation_path = raster_dir / f'landslide_source_and_runout_reafforestation_rp_{return_period}_epsg3448.tif'
    for path in [baseline_path, reafforestation_path]:
        if not path.exists():
            raise FileNotFoundError(path)

    with rasterio.open(baseline_path) as baseline_src, rasterio.open(reafforestation_path) as reafforestation_src:
        baseline = baseline_src.read(1).astype('int16')
        reafforestation = reafforestation_src.read(1).astype('int16')

    class_reduction = np.maximum(baseline - reafforestation, 0).astype('uint8')
    contribution_mask = afforestable_footprint & (class_reduction > 0)

    reafforestation_any_class_reduction |= contribution_mask
    reafforestation_max_class_reduction = np.maximum(reafforestation_max_class_reduction, class_reduction)

    status_summary = summarize_mask_by_status(
        contribution_mask,
        forest_weight=afforestable_equivalent,
        class_reduction=class_reduction,
        return_period=return_period,
        summary_type='afforestable_with_baseline_minus_reafforestation_class_reduction',
    ).rename(columns={
        'Forest_Footprint_Area_ha': 'Afforestable_Footprint_Area_ha',
        'Forest_Equivalent_Area_ha': 'Afforestable_Equivalent_Area_ha',
        'Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
        'Pct_of_Total_Forest_Footprint_Area_ha': 'Pct_of_Total_Afforestable_Footprint_Area_ha',
        'Pct_of_Total_Forest_Equivalent_Area_ha': 'Pct_of_Total_Afforestable_Equivalent_Area_ha',
        'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Pct_of_Total_Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
    })
    union_summary = summarize_union_status(status_summary.rename(columns={
        'Afforestable_Footprint_Area_ha': 'Forest_Footprint_Area_ha',
        'Afforestable_Equivalent_Area_ha': 'Forest_Equivalent_Area_ha',
        'Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha': 'Class_Reduction_Weighted_Forest_Equivalent_Area_ha',
    })).rename(columns={
        'Forest_Footprint_Area_ha': 'Afforestable_Footprint_Area_ha',
        'Forest_Equivalent_Area_ha': 'Afforestable_Equivalent_Area_ha',
        'Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
        'Pct_of_Total_Forest_Footprint_Area_ha': 'Pct_of_Total_Afforestable_Footprint_Area_ha',
        'Pct_of_Total_Forest_Equivalent_Area_ha': 'Pct_of_Total_Afforestable_Equivalent_Area_ha',
        'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Pct_of_Total_Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
    })

    reafforestation_rp_status_summaries.append(status_summary)
    reafforestation_rp_union_summaries.append(union_summary)

reafforestation_rp_status_summary = pd.concat(reafforestation_rp_status_summaries, ignore_index=True)
reafforestation_rp_union_summary = pd.concat(reafforestation_rp_union_summaries, ignore_index=True)

reafforestation_any_status_summary = summarize_mask_by_status(
    reafforestation_any_class_reduction,
    forest_weight=afforestable_equivalent,
    class_reduction=reafforestation_max_class_reduction,
    return_period='any_5_10_25_50_100',
    summary_type='afforestable_with_any_return_period_class_reduction',
).rename(columns={
    'Forest_Footprint_Area_ha': 'Afforestable_Footprint_Area_ha',
    'Forest_Equivalent_Area_ha': 'Afforestable_Equivalent_Area_ha',
    'Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
    'Pct_of_Total_Forest_Footprint_Area_ha': 'Pct_of_Total_Afforestable_Footprint_Area_ha',
    'Pct_of_Total_Forest_Equivalent_Area_ha': 'Pct_of_Total_Afforestable_Equivalent_Area_ha',
    'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Pct_of_Total_Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
})

reafforestation_any_union_summary = summarize_union_status(reafforestation_any_status_summary.rename(columns={
    'Afforestable_Footprint_Area_ha': 'Forest_Footprint_Area_ha',
    'Afforestable_Equivalent_Area_ha': 'Forest_Equivalent_Area_ha',
    'Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha': 'Class_Reduction_Weighted_Forest_Equivalent_Area_ha',
})).rename(columns={
    'Forest_Footprint_Area_ha': 'Afforestable_Footprint_Area_ha',
    'Forest_Equivalent_Area_ha': 'Afforestable_Equivalent_Area_ha',
    'Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
    'Pct_of_Total_Forest_Footprint_Area_ha': 'Pct_of_Total_Afforestable_Footprint_Area_ha',
    'Pct_of_Total_Forest_Equivalent_Area_ha': 'Pct_of_Total_Afforestable_Equivalent_Area_ha',
    'Pct_of_Total_Class_Reduction_Weighted_Forest_Equivalent_Area_ha': 'Pct_of_Total_Class_Reduction_Weighted_Afforestable_Equivalent_Area_ha',
})

reafforestation_rp_status_file = output_dir / 'reafforestation_contributing_afforestable_area_by_rp_and_protected_status_source_and_runout_grid.csv'
reafforestation_rp_union_file = output_dir / 'reafforestation_contributing_afforestable_area_by_rp_and_protected_union_status_source_and_runout_grid.csv'
reafforestation_any_status_file = output_dir / 'reafforestation_contributing_afforestable_area_any_rp_by_protected_status_source_and_runout_grid.csv'
reafforestation_any_union_file = output_dir / 'reafforestation_contributing_afforestable_area_any_rp_by_protected_union_status_source_and_runout_grid.csv'

reafforestation_rp_status_summary.to_csv(reafforestation_rp_status_file, index=False)
reafforestation_rp_union_summary.to_csv(reafforestation_rp_union_file, index=False)
reafforestation_any_status_summary.to_csv(reafforestation_any_status_file, index=False)
reafforestation_any_union_summary.to_csv(reafforestation_any_union_file, index=False)

print('Saved:', reafforestation_rp_status_file)
print('Saved:', reafforestation_rp_union_file)
print('Saved:', reafforestation_any_status_file)
print('Saved:', reafforestation_any_union_file)
display(reafforestation_any_union_summary)
display(reafforestation_rp_union_summary)

## Method metadata

In [ ]:
metadata = pd.DataFrame([
    {
        'Reference_Raster': str(reference_raster),
        'Landuse_Path': str(landuse_path),
        'Forest_Reserve_Path': str(forest_reserves_path),
        'Protected_Area_Path': str(protected_areas_path),
        'Return_Periods': ','.join(map(str, return_periods)),
        'Forest_Definition': 'forest_flood_equivalent_values > 0; forest-equivalent area weighted by forest_flood_equivalent_values',
        'Hazard_Reduction_Definition': 'deforestation source/runout class > baseline source/runout class',
        'Rasterization_All_Touched': False,
        'Protected_Areas_Include_Input_Proposed_Protected_Area_Records': bool('PROPOSED_PROTECTED_AREA' in protected_areas.get('LAYER', pd.Series(dtype=object)).astype(str).unique()),
        'Interpretation_Caveat': 'Area-based hazard-reduction attribution; avoided-EAD dollars are not causally allocated to forest polygons.',
    }
])
metadata_file = output_dir / 'landslide_protected_forest_attribution_method_metadata.csv'
metadata.to_csv(metadata_file, index=False)
print('Saved:', metadata_file)
display(metadata)